# TradeOff · coordination lab
Run with the Python (TradeOff) kernel. All company data here is synthetic. The coordinator sees submitted signals; there is no cryptographic privacy or tenant authentication.

In [ ]:
import sys, platform
from pathlib import Path
root = Path.cwd()
if not (root / "optimizer.py").exists():
    root = root.parent
if not (root / "optimizer.py").exists():
    raise RuntimeError("Open Jupyter in the TradeOff project or notebooks folder")
sys.path.insert(0, str(root))
print(sys.version)
print("Architecture:", platform.machine())
from optimizer import Request, Offer, solve, demo

## Submit limited coordination signals
Customer identities, full inventories, and detailed contracts are not collected. Penalties and offer prices ARE visible to the coordinator.

In [ ]:
requests, offers = demo()
result = solve(requests, offers)
result

## Verify the default plan

In [ ]:
assert result["objective"] == 7600
assert [d["allocated"] for d in result["decisions"]] == [600, 500]
print("Verified: all demand served for $7,600")

## Simulate a delay

In [ ]:
delayed = [offers[0], Offer("Supplier T", 300, 5, 12)]
delayed_result = solve(requests, delayed)
assert delayed_result["objective"] == 11500
delayed_result["decisions"]

## Start the frontend
Run this cell once; use the SSH tunnel in README.md to open http://localhost:8501. Stop the process with the last cell.

In [ ]:
import subprocess
if "dashboard" not in globals() or dashboard.poll() is not None:
    dashboard_log = open(root / "dashboard.log", "a")
    dashboard = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.address", "127.0.0.1", "--server.port", "8501"], cwd=root, stdout=dashboard_log, stderr=subprocess.STDOUT)
print("Dashboard PID:", dashboard.pid)

In [ ]:
dashboard.terminate()
dashboard.wait(timeout=10)
dashboard_log.close()